In [1]:
# Import required libraries
import pandas as pd
from IPython.display import display, HTML
import sys
sys.path.append('src')
from parsers.parser_factory import ParserFactory
from utils.fraud_detector import FraudDetector

In [2]:
# Set pandas display options to show full data
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# bank = 'hdfc'
# file_path = 'data/hdfc/Acct_Statement_XX0547_10052024.pdf'
# bank = 'icici'
# file_path = 'data/icici/icici_statment.pdf'
# bank = 'axis'
# file_path = 'data/axis/Axis_sample4.pdf'
bank = 'sbi'
file_path = 'data/sbi/sbi_sample2.pdf'

parser = ParserFactory.get_parser(bank)
transactions = parser.parse(file_path)
df = pd.DataFrame([t.__dict__ for t in transactions])
print(f"Parsed {len(transactions)} transactions.")

# Create a separate metadata DataFrame
metadata = parser.parse_metadata(file_path)
metadata_df = pd.DataFrame([metadata])
print("\nMetadata DataFrame:")
display(metadata_df)

# Filter transaction output columns (only if transactions were parsed)
if not df.empty:
    df = df[['date', 'narration', 'withdrawal_amt', 'deposit_amt', 'closing_balance']]
else:
    print("No transactions parsed — check file path or bank format.")

Parsed 102 transactions.


Field,Value
Bank,SBI
Account Holder,BANKIM BISWAS
Account Number,00000030348003729
Address,"VILL. BABUYIA, KOPAI, KONARPUR SAINTHIA BIRBHUM-731201, Birbhum"
Joint Holder,
Branch,BOLPUR
IFSC code,SBIN0002027
CIF No,85210484728
Account Status,
Scheme,REGULAR SB CHQ-INDIVIDUALS


In [4]:
# Display transactions as scrollable HTML table with no-wrap on date/amount columns
html = df.to_html(index=False)

css = """
<style>
  .bsa-table th, .bsa-table td { padding: 4px 12px; }
  .bsa-table td:nth-child(1),
  .bsa-table td:nth-child(3),
  .bsa-table td:nth-child(4),
  .bsa-table td:nth-child(5) { white-space: nowrap; }
</style>
"""
html = html.replace('<table ', '<table class="bsa-table" ')
display(HTML(f'{css}<div style="height:600px; overflow:auto;">{html}</div>'))

date,narration,withdrawal_amt,deposit_amt,closing_balance
01-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324410225990/Manapp 4897694162092 ur/HDFC/manappuram/NO RE-,"2,912.00",,659.89
01-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324419769809/SUMO 4897694162092 N G/SBIN/9091310929/NO RE -,200.00,,459.89
01-09-2023,BY TRANSFER- TRANSFER NEFT*ICIC0SF0002*33500033 FROM 231DC*KALPATARU 3199419044300 PROJECTS-,,49392.00,"49,851.89"
02-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324511198104/BABAN 4897695162091 C/SBIN/9593529195/NO RE-,1500.00,,"48,351.89"
02-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324514337297/UMAKA 4897695162091 NT /ICIC/8792772221/NO RE-,5000.00,,"43,351.89"
02-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324520730679/RELIAN 4897695162091 CE/CITI/jio@citiba/JIO20-,19.00,,"43,332.89"
03-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324609939018/SBI 4897696162090 CARDS/HDFC/billdeskpg/Pay-,6000.00,,"37,332.89"
03-09-2023,BY TRANSFER- TRANSFER UPI/CR/324617195477/RAKES FROM H /KKBK/yrakesh274/Payme- 4897738162095,,500.00,"37,832.89"
03-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324619452575/BHIMA 4897696162090 RAM/PYTM/paytmqr1o8/NO-,100.00,,"37,732.89"
03-09-2023,TO TRANSFER- TRANSFER TO UPI/DR/324620512427/SUMO 4897696162090 N G/SBIN/9091310929/NO RE -,1500.00,,"36,232.89"


In [5]:
# Fraud Detection Analysis
detector = FraudDetector()
report = detector.analyze(transactions, metadata, file_path=file_path)

score = report['risk_score']
level = report['risk_level']
flags = report['flags']
summary = report['summary']

# Pick colour based on risk level
colour_map = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'High': '#e67e22', 'Very High': '#e74c3c'}
colour = colour_map.get(level, '#95a5a6')

score_html = f"""
<div style="font-family:sans-serif; margin-bottom:16px;">
  <span style="font-size:18px; font-weight:bold;">Fraud Risk Score: </span>
  <span style="font-size:22px; font-weight:bold; color:{colour};">{score}/100 — {level} Risk</span>
  <span style="margin-left:24px; color:#555;">
    &nbsp;Flags: {summary['total_flags']} total
    &nbsp;|&nbsp; <span style="color:#e74c3c;">HIGH: {summary['high']}</span>
    &nbsp;|&nbsp; <span style="color:#f39c12;">MEDIUM: {summary['medium']}</span>
    &nbsp;|&nbsp; <span style="color:#3498db;">LOW: {summary['low']}</span>
  </span>
</div>
"""

if flags:
    rows = ""
    sev_colours = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#3498db'}
    for f in flags:
        sc = sev_colours.get(f['severity'], '#95a5a6')
        rows += f"""
        <tr>
          <td style="white-space:nowrap; color:{sc}; font-weight:bold; padding:4px 12px;">{f['severity']}</td>
          <td style="white-space:nowrap; padding:4px 12px;">{f['check']}</td>
          <td style="padding:4px 12px;">{f['detail']}</td>
        </tr>"""
    flags_html = f"""
    <table style="border-collapse:collapse; width:100%; font-family:sans-serif; font-size:13px;">
      <thead>
        <tr style="background:#f0f0f0;">
          <th style="padding:6px 12px; text-align:left;">Severity</th>
          <th style="padding:6px 12px; text-align:left;">Check</th>
          <th style="padding:6px 12px; text-align:left;">Detail</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>"""
else:
    flags_html = '<p style="font-family:sans-serif; color:#2ecc71;">✓ No fraud indicators detected.</p>'

display(HTML(score_html + flags_html))